In [2]:
"""
=============================================================================
  DES (Data Encryption Standard) — Full Python Implementation
  with verbose stage-by-stage output for learning/debugging
=============================================================================

  DES Overview:
  -------------
  DES is a symmetric-key block cipher that:
    • Takes a 64-bit (8-byte) plaintext block
    • Uses a 56-bit effective key (supplied as 64-bit with parity bits)
    • Produces a 64-bit ciphertext block
    • Runs 16 Feistel rounds, each using a unique 48-bit sub-key

  High-level pipeline:
    Plaintext → Initial Permutation (IP)
              → 16 Feistel Rounds (each: Expansion, XOR with sub-key,
                                         S-Box substitution, P-Box permutation)
              → Final Permutation (IP⁻¹)
              → Ciphertext
=============================================================================
"""

'\n=============================================================================\n  DES (Data Encryption Standard) — Full Python Implementation\n  with verbose stage-by-stage output for learning/debugging\n=============================================================================\n\n  DES Overview:\n  -------------\n  DES is a symmetric-key block cipher that:\n    • Takes a 64-bit (8-byte) plaintext block\n    • Uses a 56-bit effective key (supplied as 64-bit with parity bits)\n    • Produces a 64-bit ciphertext block\n    • Runs 16 Feistel rounds, each using a unique 48-bit sub-key\n\n  High-level pipeline:\n    Plaintext → Initial Permutation (IP)\n              → 16 Feistel Rounds (each: Expansion, XOR with sub-key,\n                                         S-Box substitution, P-Box permutation)\n              → Final Permutation (IP⁻¹)\n              → Ciphertext\n=============================================================================\n'

#Standard-library imports and data/key setup

In [3]:
import textwrap

# ═════════════════════════════════════════════════════════════════════════════
#  DES TABLES  (all indices are 1-based as per the original DES spec,
#               converted to 0-based inside the helper functions)
# ═════════════════════════════════════════════════════════════════════════════

# ── Initial Permutation (IP) ──────────────────────────────────────────────────
# Rearranges the 64 bits of the plaintext before the 16 rounds begin.
IP = [
    58, 50, 42, 34, 26, 18, 10, 2,
    60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6,
    64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17,  9, 1,
    59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5,
    63, 55, 47, 39, 31, 23, 15, 7,
]

# ── Inverse Initial Permutation (IP⁻¹ / FP) ──────────────────────────────────
# Undoes the IP after all 16 rounds to produce the final ciphertext.
IP_INV = [
    40, 8, 48, 16, 56, 24, 64, 32,
    39, 7, 47, 15, 55, 23, 63, 31,
    38, 6, 46, 14, 54, 22, 62, 30,
    37, 5, 45, 13, 53, 21, 61, 29,
    36, 4, 44, 12, 52, 20, 60, 28,
    35, 3, 43, 11, 51, 19, 59, 27,
    34, 2, 42, 10, 50, 18, 58, 26,
    33, 1, 41,  9, 49, 17, 57, 25,
]

# ── Expansion Permutation (E) ─────────────────────────────────────────────────
# Expands the 32-bit right half to 48 bits so it can be XORed with a sub-key.
# Some bits are deliberately repeated to provide the "expansion".
E = [
    32,  1,  2,  3,  4,  5,
     4,  5,  6,  7,  8,  9,
     8,  9, 10, 11, 12, 13,
    12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21,
    20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29,
    28, 29, 30, 31, 32,  1,
]

# ── P-Box Permutation (P) ─────────────────────────────────────────────────────
# Permutes the 32-bit output of the S-Boxes within each Feistel round.
P = [
    16,  7, 20, 21,
    29, 12, 28, 17,
     1, 15, 23, 26,
     5, 18, 31, 10,
     2,  8, 24, 14,
    32, 27,  3,  9,
    19, 13, 30,  6,
    22, 11,  4, 25,
]

# ── Permuted Choice 1 (PC-1) ──────────────────────────────────────────────────
# Selects 56 bits from the 64-bit key (drops the 8 parity bits) and permutes.
PC1 = [
    57, 49, 41, 33, 25, 17,  9,
     1, 58, 50, 42, 34, 26, 18,
    10,  2, 59, 51, 43, 35, 27,
    19, 11,  3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15,
     7, 62, 54, 46, 38, 30, 22,
    14,  6, 61, 53, 45, 37, 29,
    21, 13,  5, 28, 20, 12,  4,
]

# ── Permuted Choice 2 (PC-2) ──────────────────────────────────────────────────
# Selects and permutes 48 bits from the 56-bit shifted key halves to form
# each round's sub-key.
PC2 = [
    14, 17, 11, 24,  1,  5,
     3, 28, 15,  6, 21, 10,
    23, 19, 12,  4, 26,  8,
    16,  7, 27, 20, 13,  2,
    41, 52, 31, 37, 47, 55,
    30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53,
    46, 42, 50, 36, 29, 32,
]

# ── Left-shift schedule ───────────────────────────────────────────────────────
# Number of left circular shifts applied to each key half per round.
SHIFT_SCHEDULE = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

# ── S-Boxes ───────────────────────────────────────────────────────────────────
# 8 substitution boxes, each mapping 6 bits → 4 bits.
# Row  = outer two bits (bits 1 & 6 of the 6-bit group)
# Col  = inner four bits (bits 2-5)
S_BOXES = [
    # S1
    [
        [14,  4, 13,  1,  2, 15, 11,  8,  3, 10,  6, 12,  5,  9,  0,  7],
        [ 0, 15,  7,  4, 14,  2, 13,  1, 10,  6, 12, 11,  9,  5,  3,  8],
        [ 4,  1, 14,  8, 13,  6,  2, 11, 15, 12,  9,  7,  3, 10,  5,  0],
        [15, 12,  8,  2,  4,  9,  1,  7,  5, 11,  3, 14, 10,  0,  6, 13],
    ],
    # S2
    [
        [15,  1,  8, 14,  6, 11,  3,  4,  9,  7,  2, 13, 12,  0,  5, 10],
        [ 3, 13,  4,  7, 15,  2,  8, 14, 12,  0,  1, 10,  6,  9, 11,  5],
        [ 0, 14,  7, 11, 10,  4, 13,  1,  5,  8, 12,  6,  9,  3,  2, 15],
        [13,  8, 10,  1,  3, 15,  4,  2, 11,  6,  7, 12,  0,  5, 14,  9],
    ],
    # S3
    [
        [10,  0,  9, 14,  6,  3, 15,  5,  1, 13, 12,  7, 11,  4,  2,  8],
        [13,  7,  0,  9,  3,  4,  6, 10,  2,  8,  5, 14, 12, 11, 15,  1],
        [13,  6,  4,  9,  8, 15,  3,  0, 11,  1,  2, 12,  5, 10, 14,  7],
        [ 1, 10, 13,  0,  6,  9,  8,  7,  4, 15, 14,  3, 11,  5,  2, 12],
    ],
    # S4
    [
        [ 7, 13, 14,  3,  0,  6,  9, 10,  1,  2,  8,  5, 11, 12,  4, 15],
        [13,  8, 11,  5,  6, 15,  0,  3,  4,  7,  2, 12,  1, 10, 14,  9],
        [10,  6,  9,  0, 12, 11,  7, 13, 15,  1,  3, 14,  5,  2,  8,  4],
        [ 3, 15,  0,  6, 10,  1, 13,  8,  9,  4,  5, 11, 12,  7,  2, 14],
    ],
    # S5
    [
        [ 2, 12,  4,  1,  7, 10, 11,  6,  8,  5,  3, 15, 13,  0, 14,  9],
        [14, 11,  2, 12,  4,  7, 13,  1,  5,  0, 15, 10,  3,  9,  8,  6],
        [ 4,  2,  1, 11, 10, 13,  7,  8, 15,  9, 12,  5,  6,  3,  0, 14],
        [11,  8, 12,  7,  1, 14,  2, 13,  6, 15,  0,  9, 10,  4,  5,  3],
    ],
    # S6
    [
        [12,  1, 10, 15,  9,  2,  6,  8,  0, 13,  3,  4, 14,  7,  5, 11],
        [10, 15,  4,  2,  7, 12,  9,  5,  6,  1, 13, 14,  0, 11,  3,  8],
        [ 9, 14, 15,  5,  2,  8, 12,  3,  7,  0,  4, 10,  1, 13, 11,  6],
        [ 4,  3,  2, 12,  9,  5, 15, 10, 11, 14,  1,  7,  6,  0,  8, 13],
    ],
    # S7
    [
        [ 4, 11,  2, 14, 15,  0,  8, 13,  3, 12,  9,  7,  5, 10,  6,  1],
        [13,  0, 11,  7,  4,  9,  1, 10, 14,  3,  5, 12,  2, 15,  8,  6],
        [ 1,  4, 11, 13, 12,  3,  7, 14, 10, 15,  6,  8,  0,  5,  9,  2],
        [ 6, 11, 13,  8,  1,  4, 10,  7,  9,  5,  0, 15, 14,  2,  3, 12],
    ],
    # S8
    [
        [13,  2,  8,  4,  6, 15, 11,  1, 10,  9,  3, 14,  5,  0, 12,  7],
        [ 1, 15, 13,  8, 10,  3,  7,  4, 12,  5,  6, 11,  0, 14,  9,  2],
        [ 7, 11,  4,  1,  9, 12, 14,  2,  0,  6, 10, 13, 15,  3,  5,  8],
        [ 2,  1, 14,  7,  4, 10,  8, 13, 15, 12,  9,  0,  3,  5,  6, 11],
    ],
]

#UTILITY / HELPER FUNCTIONS

In [4]:
def bytes_to_bits(data: bytes) -> list[int]:
    """Convert a bytes object to a flat list of individual bits (MSB first).

    Example: b'\\x80' → [1, 0, 0, 0, 0, 0, 0, 0]
    """
    bits = []
    for byte in data:
        for i in range(7, -1, -1):   # iterate from bit 7 (MSB) down to bit 0
            bits.append((byte >> i) & 1)
    return bits


def bits_to_bytes(bits: list[int]) -> bytes:
    """Convert a flat list of bits back to a bytes object.

    The list length must be a multiple of 8.
    """
    result = bytearray()
    for i in range(0, len(bits), 8):
        byte_bits = bits[i:i + 8]
        byte_val = 0
        for bit in byte_bits:
            byte_val = (byte_val << 1) | bit
        result.append(byte_val)
    return bytes(result)


def permute(bits: list[int], table: list[int]) -> list[int]:
    """Apply a permutation / selection table to a bit list.

    The table uses 1-based indices (as in the DES spec), so we subtract 1.
    """
    return [bits[t - 1] for t in table]


def xor_bits(a: list[int], b: list[int]) -> list[int]:
    """XOR two equal-length bit lists element-wise."""
    return [x ^ y for x, y in zip(a, b)]


def left_rotate(bits: list[int], n: int) -> list[int]:
    """Circularly left-shift a bit list by n positions."""
    return bits[n:] + bits[:n]


def bits_to_hex(bits: list[int]) -> str:
    """Format a bit list as a compact hex string for readable output."""
    return bits_to_bytes(bits).hex().upper()


def format_bits(bits: list[int], group: int = 8) -> str:
    """Pretty-print a bit list with spaces every `group` bits."""
    s = "".join(str(b) for b in bits)
    return " ".join(textwrap.wrap(s, group))

#KEY SCHEDULE  — generate 16 × 48-bit sub-keys from the 64-bit master key

In [5]:
def generate_subkeys(key_bytes: bytes, verbose: bool = True) -> list[list[int]]:
    """Derive all 16 round sub-keys (each 48 bits) from the 64-bit key.

    Steps:
      1. Convert key to 64 bits
      2. Apply PC-1 → 56-bit permuted key (drops 8 parity bits)
      3. Split into two 28-bit halves: C and D
      4. For each round i:
           a. Circularly left-shift C and D by SHIFT_SCHEDULE[i]
           b. Concatenate shifted C and D
           c. Apply PC-2 to select/permute 48 bits → sub-key Kᵢ
    """
    print("\n" + "═" * 70)
    print("  KEY SCHEDULE — Generating 16 Sub-Keys")
    print("═" * 70)

    # ── Step 1: key bytes → 64-bit list ──────────────────────────────────────
    key_bits = bytes_to_bits(key_bytes)
    print(f"\n[1] Original 64-bit key (hex): {key_bytes.hex().upper()}")
    print(f"    Bits: {format_bits(key_bits)}")

    # ── Step 2: Apply PC-1 (64 → 56 bits, parity bits discarded) ─────────────
    key_56 = permute(key_bits, PC1)
    print(f"\n[2] After PC-1 (56 bits, parity stripped):")
    print(f"    {format_bits(key_56, 7)}")

    # ── Step 3: Split into C₀ (28 bits) and D₀ (28 bits) ─────────────────────
    C = key_56[:28]
    D = key_56[28:]
    print(f"\n[3] C₀ (left 28 bits) : {format_bits(C, 7)}")
    print(f"    D₀ (right 28 bits): {format_bits(D, 7)}")

    subkeys: list[list[int]] = []

    print(f"\n[4] Generating sub-keys K1–K16 (each 48 bits):\n")
    print(f"    {'Round':<8} {'Shifts':<8} {'Kᵢ (hex)':<14} {'Kᵢ (bits)'}")
    print(f"    {'-'*8} {'-'*8} {'-'*14} {'-'*50}")

    for i in range(16):
        shift = SHIFT_SCHEDULE[i]

        # ── Step 4a: circular left-shift both halves ──────────────────────────
        C = left_rotate(C, shift)
        D = left_rotate(D, shift)

        # ── Step 4b: concatenate ──────────────────────────────────────────────
        CD = C + D           # 56 bits total

        # ── Step 4c: apply PC-2 to produce 48-bit sub-key ─────────────────────
        subkey = permute(CD, PC2)
        subkeys.append(subkey)

        if verbose:
            ki_hex = bits_to_hex(subkey + [0] * 16)  # pad to 8 bytes for hex display
            ki_bits = format_bits(subkey, 6)
            print(f"    K{i+1:<7} {shift:<8} {ki_hex:<14} {ki_bits}")

    print(f"\n  ✓ All 16 sub-keys generated successfully.")
    return subkeys

#FEISTEL FUNCTION  f(R, K)

In [6]:
def feistel_function(R: list[int], K: list[int], round_num: int,
                     verbose: bool = True) -> list[int]:
    """The DES Feistel / 'f' function applied to a 32-bit half-block R
    and a 48-bit sub-key K.

    Steps:
      1. Expansion (E):   32 bits → 48 bits
      2. XOR with sub-key K:  48 bits
      3. S-Box substitution:  48 bits → 32 bits  (8 S-boxes, 6-bits-in 4-bits-out)
      4. P-Box permutation:   32 bits → 32 bits
    """
    prefix = f"    [R{round_num}]"

    # ── Step 1: Expansion E (32 → 48 bits) ───────────────────────────────────
    R_expanded = permute(R, E)
    if verbose:
        print(f"{prefix} E(R):      {bits_to_hex(R_expanded + [0]*16)}"
              f"  ({len(R_expanded)} bits)")

    # ── Step 2: XOR with sub-key ──────────────────────────────────────────────
    xored = xor_bits(R_expanded, K)
    if verbose:
        print(f"{prefix} XOR K:     {bits_to_hex(xored + [0]*16)}"
              f"  ({len(xored)} bits)")

    # ── Step 3: S-Box substitution (48 → 32 bits) ────────────────────────────
    # Split 48-bit xored result into 8 groups of 6 bits.
    # For each group:
    #   row = (bit0 << 1) | bit5   — first and last bits
    #   col = bits 1-4 as a 4-bit number
    sbox_output: list[int] = []
    if verbose:
        print(f"{prefix} S-Box substitution:")

    for s in range(8):                      # s = S-Box index 0..7
        group = xored[s * 6:(s + 1) * 6]   # 6-bit chunk for S-Box s+1
        row = (group[0] << 1) | group[5]   # row index: bits 0 and 5
        col = (group[1] << 3) | (group[2] << 2) | (group[3] << 1) | group[4]  # bits 1-4
        val = S_BOXES[s][row][col]          # 4-bit substitution value

        # Convert the 4-bit integer back to 4 bits (MSB first)
        out_bits = [(val >> (3 - j)) & 1 for j in range(4)]
        sbox_output.extend(out_bits)

        if verbose:
            group_str = "".join(str(b) for b in group)
            out_str   = "".join(str(b) for b in out_bits)
            print(f"         S{s+1}: [{group_str}] row={row} col={col:02d}"
                  f" → val={val:2d} → [{out_str}]")

    if verbose:
        print(f"{prefix} Post-S-Box: {bits_to_hex(sbox_output + [0]*0)}"
              f"  ({len(sbox_output)} bits)")

    # ── Step 4: P-Box permutation (32 → 32 bits) ─────────────────────────────
    result = permute(sbox_output, P)
    if verbose:
        print(f"{prefix} After P:   {bits_to_hex(result)}"
              f"  ({len(result)} bits)")

    return result

#DES BLOCK CIPHER  — encrypt or decrypt a single 64-bit block

In [7]:
def des_block(block: bytes, key: bytes,
              encrypt: bool = True,
              verbose: bool = True) -> bytes:
    """Encrypt or decrypt a single 8-byte (64-bit) block using DES.

    Parameters
    ----------
    block   : 8 bytes of plaintext (encrypt=True) or ciphertext (encrypt=False)
    key     : 8-byte DES key (64 bits; 8 parity bits are ignored internally)
    encrypt : True → encryption mode, False → decryption mode
              (decryption uses sub-keys in reverse order K16 … K1)
    verbose : print detailed intermediate state at every step

    Returns
    -------
    8 bytes of ciphertext (or plaintext, if decrypting)
    """
    mode = "ENCRYPTION" if encrypt else "DECRYPTION"
    print("\n" + "═" * 70)
    print(f"  DES {mode} — Processing 64-bit Block")
    print("═" * 70)

    # ── Step 0: Generate all 16 sub-keys ─────────────────────────────────────
    subkeys = generate_subkeys(key, verbose=verbose)

    # Decryption uses sub-keys in reverse order (K16 → K1)
    if not encrypt:
        subkeys = subkeys[::-1]
        print("\n  [Decryption] Sub-key order reversed: K16 → K1")

    # ── Step 1: Convert plaintext block to 64 bits ────────────────────────────
    block_bits = bytes_to_bits(block)
    print(f"\n{'PLAINTEXT' if encrypt else 'CIPHERTEXT'} block : "
          f"{block.hex().upper()}  →  {format_bits(block_bits)}")

    # ── Step 2: Initial Permutation (IP) ──────────────────────────────────────
    permuted = permute(block_bits, IP)
    print(f"\n[IP ] After Initial Permutation:")
    print(f"      {format_bits(permuted)}  (hex: {bits_to_hex(permuted)})")

    # ── Step 3: Split into Left (L₀) and Right (R₀) halves ───────────────────
    L = permuted[:32]
    R = permuted[32:]
    print(f"\n[L₀ ] {format_bits(L)}  (hex: {bits_to_hex(L)})")
    print(f"[R₀ ] {format_bits(R)}  (hex: {bits_to_hex(R)})")

    # ── Step 4: 16 Feistel rounds ─────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  16 FEISTEL ROUNDS")
    print(f"{'─'*70}")

    for i in range(16):
        print(f"\n  ┌─ Round {i+1:2d} {'(sub-key K'+str(i+1 if encrypt else 16-i)+')'} "
              + "─" * 40)
        print(f"  │ L_in = {bits_to_hex(L)}   R_in = {bits_to_hex(R)}")

        # Feistel structure:
        #   L_{i+1} = R_i
        #   R_{i+1} = L_i ⊕ f(R_i, K_{i+1})
        f_result = feistel_function(R, subkeys[i], i + 1, verbose=verbose)
        new_R = xor_bits(L, f_result)   # L becomes old R; R becomes L XOR f(R, K)
        new_L = R

        L, R = new_L, new_R

        print(f"  │ L_out= {bits_to_hex(L)}   R_out= {bits_to_hex(R)}")
        print(f"  └{'─'*55}")

    # ── Step 5: Final swap — combine as R₁₆ ∥ L₁₆  (note: R before L) ────────
    # After 16 rounds the standard DES spec concatenates R16 ∥ L16
    # (i.e. the halves are NOT swapped at the last round).
    pre_output = R + L          # right half first, then left
    print(f"\n[Pre-IP⁻¹] R₁₆∥L₁₆ : {format_bits(pre_output)}")
    print(f"           (hex)    : {bits_to_hex(pre_output)}")

    # ── Step 6: Final / Inverse Permutation (IP⁻¹) ────────────────────────────
    output_bits = permute(pre_output, IP_INV)
    output_bytes = bits_to_bytes(output_bits)

    print(f"\n[FP ] After Final Permutation (IP⁻¹):")
    print(f"      {format_bits(output_bits)}")
    print(f"\n{'CIPHERTEXT' if encrypt else 'PLAINTEXT'} block : "
          f"{output_bytes.hex().upper()}")
    print("═" * 70)

    return output_bytes

#SIMPLE PADDING UTILITIES (PKCS#7 style, restricted to DES 8-byte blocks)

In [8]:

def pad(data: bytes) -> bytes:
    """Pad data to a multiple of 8 bytes using PKCS#7 padding.

    The padding byte value equals the number of padding bytes added.
    Example: b'HELLO' (5 bytes) → b'HELLO\\x03\\x03\\x03'
    """
    pad_len = 8 - (len(data) % 8)
    return data + bytes([pad_len] * pad_len)


def unpad(data: bytes) -> bytes:
    """Remove PKCS#7 padding from decrypted data."""
    pad_len = data[-1]
    return data[:-pad_len]

#  HIGH-LEVEL DES ECB  — encrypt / decrypt arbitrary-length messages
#  (ECB mode: each 8-byte block is encrypted independently)

In [9]:
def des_ecb_encrypt(plaintext: bytes, key: bytes,
                    verbose_first_block: bool = True) -> bytes:
    """Encrypt plaintext with DES in ECB mode.

    For readability, verbose output is shown only for the first block by default.
    """
    padded = pad(plaintext)
    ciphertext = b""
    num_blocks = len(padded) // 8

    print(f"\n{'='*70}")
    print(f"  DES-ECB ENCRYPTION")
    print(f"  Plaintext  : {plaintext!r}  ({len(plaintext)} bytes)")
    print(f"  Padded     : {padded.hex().upper()}  ({len(padded)} bytes, {num_blocks} block(s))")
    print(f"  Key        : {key.hex().upper()}")
    print(f"{'='*70}")

    for blk_idx in range(num_blocks):
        block = padded[blk_idx * 8:(blk_idx + 1) * 8]
        verbose = verbose_first_block and (blk_idx == 0)
        if not verbose:
            print(f"\n  [Block {blk_idx+1}/{num_blocks}] {block.hex().upper()} → ", end="")
        enc_block = des_block(block, key, encrypt=True, verbose=verbose)
        if not verbose:
            print(enc_block.hex().upper())
        ciphertext += enc_block

    print(f"\n  ✓ Final ciphertext : {ciphertext.hex().upper()}")
    return ciphertext


def des_ecb_decrypt(ciphertext: bytes, key: bytes,
                    verbose_first_block: bool = True) -> bytes:
    """Decrypt ciphertext with DES in ECB mode."""
    plaintext = b""
    num_blocks = len(ciphertext) // 8

    print(f"\n{'='*70}")
    print(f"  DES-ECB DECRYPTION")
    print(f"  Ciphertext : {ciphertext.hex().upper()}  ({len(ciphertext)} bytes, {num_blocks} block(s))")
    print(f"  Key        : {key.hex().upper()}")
    print(f"{'='*70}")

    for blk_idx in range(num_blocks):
        block = ciphertext[blk_idx * 8:(blk_idx + 1) * 8]
        verbose = verbose_first_block and (blk_idx == 0)
        if not verbose:
            print(f"\n  [Block {blk_idx+1}/{num_blocks}] {block.hex().upper()} → ", end="")
        dec_block = des_block(block, key, encrypt=False, verbose=verbose)
        if not verbose:
            print(dec_block.hex().upper())
        plaintext += dec_block

    plaintext = unpad(plaintext)
    print(f"\n  ✓ Final plaintext  : {plaintext!r}")
    return plaintext

#Demo Test

In [10]:
if __name__ == "__main__":

    # ── Test Vector 1: NIST-style single block ────────────────────────────────
    print("\n" + "#" * 70)
    print("#  TEST VECTOR 1 — Classic DES single block (verbose)")
    print("#" * 70)

    key1       = bytes.fromhex("133457799BBCDFF1")   # 64-bit key
    plaintext1 = bytes.fromhex("0123456789ABCDEF")   # 64-bit block

    print(f"\n  Plaintext  (hex): {plaintext1.hex().upper()}")
    print(f"  Key        (hex): {key1.hex().upper()}")

    ciphertext1 = des_block(plaintext1, key1, encrypt=True,  verbose=True)
    recovered1  = des_block(ciphertext1, key1, encrypt=False, verbose=True)

    print(f"\n  ── Verification ──")
    print(f"  Original  : {plaintext1.hex().upper()}")
    print(f"  Encrypted : {ciphertext1.hex().upper()}")
    print(f"  Decrypted : {recovered1.hex().upper()}")
    assert plaintext1 == recovered1, "Round-trip failed!"
    print(f"  ✓ Round-trip check PASSED")

    # ── Test Vector 2: Multi-block ECB (abbreviated output) ───────────────────
    print("\n\n" + "#" * 70)
    print("#  TEST VECTOR 2 — Multi-block ECB (verbose only on first block)")
    print("#" * 70)

    key2       = b"SecretK!"                # 8-byte ASCII key
    plaintext2 = b"Hello, DES World!"      # 17 bytes → needs 3 blocks after padding

    ciphertext2 = des_ecb_encrypt(plaintext2, key2, verbose_first_block=True)
    recovered2  = des_ecb_decrypt(ciphertext2, key2, verbose_first_block=True)

    print(f"\n  ── Verification ──")
    print(f"  Original  : {plaintext2!r}")
    print(f"  Decrypted : {recovered2!r}")
    assert plaintext2 == recovered2, "Round-trip failed!"
    print(f"  ✓ Round-trip check PASSED")

    # ── Test Vector 3: Known-answer test (KAT) ────────────────────────────────
    print("\n\n" + "#" * 70)
    print("#  TEST VECTOR 3 — Known-Answer Test (no verbose block output)")
    print("#" * 70)

    # DES KAT from FIPS-81 / original specification
    kat_key   = bytes.fromhex("0E329232EA6D0D73")
    kat_plain = bytes.fromhex("8787878787878787")
    kat_exp   = bytes.fromhex("0000000000000000")

    kat_result = des_block(kat_plain, kat_key, encrypt=True, verbose=False)
    print(f"\n  Key       : {kat_key.hex().upper()}")
    print(f"  Plaintext : {kat_plain.hex().upper()}")
    print(f"  Expected  : {kat_exp.hex().upper()}")
    print(f"  Got       : {kat_result.hex().upper()}")
    if kat_result == kat_exp:
        print("  ✓ KAT PASSED")
    else:
        print("  ✗ KAT FAILED — check S-box / table transcription")


######################################################################
#  TEST VECTOR 1 — Classic DES single block (verbose)
######################################################################

  Plaintext  (hex): 0123456789ABCDEF
  Key        (hex): 133457799BBCDFF1

══════════════════════════════════════════════════════════════════════
  DES ENCRYPTION — Processing 64-bit Block
══════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
  KEY SCHEDULE — Generating 16 Sub-Keys
══════════════════════════════════════════════════════════════════════

[1] Original 64-bit key (hex): 133457799BBCDFF1
    Bits: 00010011 00110100 01010111 01111001 10011011 10111100 11011111 11110001

[2] After PC-1 (56 bits, parity stripped):
    1111000 0110011 0010101 0101111 0101010 1011001 1001111 0001111

[3] C₀ (left 28 bits) : 1111000 0110011 0010101 0101111
    D₀ (right 28 bits): 0101010 1011001 1001111 0001111

[4